Contruimos el pipeline completo que siga el siguiente flujo:

RAW DATA
↓
Cleaning
↓
Feature Engineering
↓
Train/Test Split
↓
SMOTE
↓
Preprocessing
↓
XGBoost
↓
Guardar pipeline completo

# Imports

In [17]:
from pathlib import Path

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from imblearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from sklearn.linear_model import LogisticRegression 
from sklearn.ensemble import ( 
    RandomForestClassifier 
)

from xgboost import XGBClassifier

from imblearn.over_sampling import SMOTE

# Cargamos el data set - raw

In [2]:
DATA_PATH = Path(
    "../data/raw/01-hotel_bookings.csv"
)

df = pd.read_csv(DATA_PATH)

print(df.shape)
df.head()

(119390, 32)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,01-07-15
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,01-07-15
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,02-07-15
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,02-07-15
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,03-07-15


In [3]:
df.columns

Index(['hotel', 'is_canceled', 'lead_time', 'arrival_date_year',
       'arrival_date_month', 'arrival_date_week_number',
       'arrival_date_day_of_month', 'stays_in_weekend_nights',
       'stays_in_week_nights', 'adults', 'children', 'babies', 'meal',
       'country', 'market_segment', 'distribution_channel',
       'is_repeated_guest', 'previous_cancellations',
       'previous_bookings_not_canceled', 'reserved_room_type',
       'assigned_room_type', 'booking_changes', 'deposit_type', 'agent',
       'company', 'days_in_waiting_list', 'customer_type', 'adr',
       'required_car_parking_spaces', 'total_of_special_requests',
       'reservation_status', 'reservation_status_date'],
      dtype='str')

# Data cleaning

In [4]:
# Eliminar columnas leakage
cols_drop = [
    "reservation_status",
    "reservation_status_date",
    "agent",
    "company",
    "country"
]

df.drop(columns=cols_drop, inplace=True)

# Eliminar reservas sin personas
mask_people = (
    (df["adults"] == 0) &
    (df["children"].fillna(0) == 0) &
    (df["babies"] == 0)
)

df = df[~mask_people]

# Eliminar nulos en children
df.dropna(subset=["children"], inplace=True)

# Corregir tipo
df["children"] = df["children"].astype(int)

# Eliminar children=10
df = df[df["children"] != 10]

# Eliminar adr negativo
df = df[df["adr"] >= 0]

# Capear adr
p99 = df["adr"].quantile(0.99)

df["adr"] = df["adr"].clip(upper=p99)

print(df.shape)

(119204, 27)


## FEATURE ENGINEERING

In [5]:
# Total nights
df["total_nights"] = (
    df["stays_in_week_nights"] +
    df["stays_in_weekend_nights"]
)

# Total guests
df["total_guests"] = (
    df["adults"] +
    df["children"] +
    df["babies"]
)

# High season
high_season = ["July", "August"]

df["is_high_season"] = (
    df["arrival_date_month"]
    .isin(high_season)
    .astype(int)
)

# Log transforms
df["adr_log"] = np.log1p(df["adr"])
df["lead_time_log"] = np.log1p(df["lead_time"])

## ELIMINAR VARIABLES REDUNDANTES

In [6]:
df.drop(columns=[
    "adults",
    "children",
    "babies",
    "adr",
    "lead_time",
    "stays_in_week_nights",
    "stays_in_weekend_nights"
], inplace=True)

## SPLIT

In [7]:
X = df.drop("is_canceled", axis=1)
y = df["is_canceled"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

## COLUMNAS

In [8]:
num_cols = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

cat_cols = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print(num_cols)
print(cat_cols)

['arrival_date_year', 'arrival_date_week_number', 'arrival_date_day_of_month', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'booking_changes', 'days_in_waiting_list', 'required_car_parking_spaces', 'total_of_special_requests', 'total_nights', 'total_guests', 'is_high_season', 'adr_log', 'lead_time_log']
['hotel', 'arrival_date_month', 'meal', 'market_segment', 'distribution_channel', 'reserved_room_type', 'assigned_room_type', 'deposit_type', 'customer_type']


C:\Users\jorge\AppData\Local\Temp\ipykernel_37528\4184047617.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(


## PREPROCESSOR

In [9]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols)
])

## PREPROCESS TRAIN

In [10]:
X_train_t = preprocessor.fit_transform(X_train)

X_test_t = preprocessor.transform(X_test)

## SMOTE

In [11]:
smote = SMOTE(random_state=42)

X_train_bal, y_train_bal = smote.fit_resample(
    X_train_t,
    y_train
)

print(X_train_bal.shape)

c:\Users\jorge\miniforge3\envs\dp261-g2\Lib\site-packages\threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


(120016, 73)


## MODELOS

In [19]:
models = { 
    "random_forest": RandomForestClassifier( 
        n_estimators=100,
        random_state=42,
        n_jobs=-1 ),
    "xgboost": XGBClassifier(
        n_estimators=300, 
        max_depth=10, 
        learning_rate=0.2, 
        colsample_bytree=0.7, 
        eval_metric="logloss", 
        random_state=42 
    ) 
}

## TRAINING LOOP

In [20]:
results = [] 
MODELS_DIR = Path("../models") 
MODELS_DIR.mkdir( 
    exist_ok=True 
) 

for name, clf in models.items():
    print(f"\nTraining {name}...") 
    
    pipeline = Pipeline([ 
        ("preprocessor", preprocessor), 
        ("smote", SMOTE(random_state=42)), 
        ("clf", clf) 
    ]) 
    
    pipeline.fit( 
        X_train, 
        y_train 
    ) 
    
    y_pred = pipeline.predict( 
        X_test 
    ) 
    
    y_proba = pipeline.predict_proba( 
        X_test 
        )[:,1] 
        
    metrics = { 
        "Model": name, 
        "Accuracy": accuracy_score( 
            y_test, 
            y_pred 
        ), 
            
        "Recall": recall_score( 
            y_test, 
            y_pred 
        ), 
            
        "F1": f1_score( 
            y_test, 
            y_pred 
        ), 
            
        "AUC": roc_auc_score( 
            y_test, 
            y_proba 
        ) 
    } 
        
    results.append(metrics) 
    joblib.dump( 
        pipeline, 
        MODELS_DIR / f"{name}.pkl" 
    )


Training random_forest...

Training xgboost...


## RESULTADOS

In [22]:
results_df = pd.DataFrame( 
      results 
) 

results_df = results_df.sort_values( 
      "AUC", 
      ascending=False 
) 

print(results_df) 

results_df.to_csv( 
      MODELS_DIR / "metrics.csv", 
      index=False 
)

           Model  Accuracy    Recall        F1       AUC
1        xgboost  0.869007  0.793529  0.817911  0.935449
0  random_forest  0.871104  0.793981  0.820387  0.935304
